# 03 - Baseline Models

Before touching anything ML-based, I want a floor to compare against. Three baselines,
each one adding a bit more signal than the last:

1. **Popularity** — same top-N products for every user, no personalization at all
2. **Reorder-based popularity** — same idea, but ranked by reorder rate instead of raw volume
3. **Personalized frequency** — each user's own most-bought products

All three, plus every model after this, get evaluated the same way: for each user with a
`train` order, generate Top-K recommendations from their `prior` history, then check how
many of the actual `train` basket items show up in those K recommendations. That's
Precision@K and Recall@K. Using a shared evaluation function means the numbers are directly
comparable once collaborative filtering and the hybrid model show up later.

Logic in `src/baseline_models.py`.

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd

from data_processing import load_raw_data
from eda import build_transactions
from baseline_models import (
    PopularityModel, ReorderPopularityModel, PersonalizedFrequencyModel,
    get_eval_users, evaluate_model
)

## Load data

Reading straight from the processed parquet if it's already there from the EDA notebook —
no reason to redo the 34M-row join again.

In [2]:
processed_path = Path.cwd().parent / "data" / "processed" / "transactions.parquet"

if processed_path.exists():
    txn = pd.read_parquet(processed_path)
else:
    data = load_raw_data()
    txn = build_transactions(data)

print(txn.shape)

(33819106, 15)


## Set up evaluation users

These are the ~131k users who have a labeled `train` basket — the actual next order we're
trying to predict. The `test` users have no visible ground truth (that's Kaggle's holdout),
so they're not usable for offline scoring.

In [3]:
eval_users = get_eval_users(txn)
print("users with a labeled train basket:", len(eval_users))
eval_users.head()

users with a labeled train basket: 131209


,user_id,true_products
0,1,"{196, 26405, 27845, 46149, 13032, 39657, 26088..."
1,2,"{24838, 11913, 45066, 31883, 48523, 38547, 248..."
2,5,"{40706, 21413, 20843, 48204, 21616, 19057, 201..."
3,7,"{17638, 29894, 47272, 45066, 13198, 37999, 408..."
4,8,"{27104, 15937, 5539, 41540, 31717, 48230, 2224..."


## Model 1: Popularity

No personalization — everyone gets the same list, ranked by raw purchase count. This is the
floor. If a personalized model can't beat this, it's not worth the added complexity.

In [4]:
pop_model = PopularityModel(top_n=50).fit(txn)
print("top 10 overall:", pop_model.ranked_products_[:10])

top 10 overall: [24852, 13176, 21137, 21903, 47209, 47766, 47626, 16797, 26209, 27845]


## Model 2: Reorder-based popularity

Purchase counts are heavily right-skewed across the catalog, so reorder rate on its own isn't
a safe ranking signal at low volume — a handful of loyal buyers can push an obscure product
to a 0.9+ reorder rate, while a genuine staple like bananas or milk sits at a "merely" 0.7-0.85
despite carrying real volume. The cell below looks at the purchase-count distribution to pick a
sensible floor before ranking, then scores products on `purchases × reorder_rate` rather than
reorder rate alone, so a product needs both scale and loyalty to rank highly.

In [5]:
prior_txn = txn[txn["eval_set"] == "prior"]
purchase_counts = prior_txn.groupby("product_id").size()

print(purchase_counts.describe())
for pct in [50, 75, 90, 95, 99]:
    print(f"{pct}th percentile: {purchase_counts.quantile(pct/100):.0f}")

count     49677.000000
mean        652.907563
std        4792.114416
min           1.000000
25%          17.000000
50%          60.000000
75%         260.000000
max      472565.000000
dtype: float64
50th percentile: 60
75th percentile: 260
90th percentile: 1021
95th percentile: 2286
99th percentile: 9931


In [6]:
min_purchases = int(purchase_counts.quantile(0.99))
print("min_purchases floor:", min_purchases)

reorder_model = ReorderPopularityModel(top_n=50, min_purchases=min_purchases).fit(txn)
print("top 10 by reorder score:", reorder_model.ranked_products_[:10])

min_purchases floor: 9931
top 10 by reorder score: [24852, 13176, 21137, 21903, 47209, 47766, 27845, 47626, 27966, 16797]


## Model 3: Personalized frequency

Each user's own most-bought products, ranked by how often they've bought them (ties broken
toward more recent purchases). New/cold-start users with no prior history fall back to the
Model 1 popularity list, since there's no personal signal to rank for them yet.

In [7]:
pf_model = PersonalizedFrequencyModel(top_n=50).fit(txn)

sample_user = eval_users.iloc[0]["user_id"]
print(f"recs for user {sample_user}:", pf_model.recommend(sample_user)[:10])

recs for user 1: [196, 12427, 10258, 25133, 13032, 46149, 49235, 13176, 26405, 26088]


## Evaluate all three

Precision@10 and Recall@10, same eval_users, same K, so the comparison is apples-to-apples.

In [8]:
K = 10

results = {
    "Popularity": evaluate_model(pop_model.recommend, eval_users, k=K),
    "Reorder Popularity": evaluate_model(reorder_model.recommend, eval_users, k=K),
    "Personalized Frequency": evaluate_model(pf_model.recommend, eval_users, k=K),
}

results_df = pd.DataFrame(results).T
results_df

,k,n_users_evaluated,precision_at_k,recall_at_k
Popularity,10.0,131209.0,0.072522,0.069843
Reorder Popularity,10.0,131209.0,0.072151,0.069616
Personalized Frequency,10.0,131209.0,0.283836,0.329784


### Findings

The validated dataset provides a reliable foundation for establishing recommendation baselines. The full dataset contains **206,209 users, 3.4M orders, and 49,688 products**, with no orphaned keys, duplicate order IDs, or invalid product/category relationships identified during validation.

The EDA indicates several important behavioral signals that shape the baseline strategy:

* **Repeat purchasing is a strong signal.** The overall reorder rate is approximately **59%**, indicating that a substantial proportion of purchases are repeat purchases. This makes historical purchase frequency particularly relevant for personalized recommendations.

* **Customer behavior varies considerably.** Users place an average of **15.6 prior orders**, with a median of **9 orders**. The average basket contains approximately **10 items**, while the median is **8 items**. Customer-level reorder rates also show substantial variation, supporting the use of individualized purchase history rather than relying exclusively on global popularity.

* **Product popularity is highly concentrated.** Products such as **Banana, Bag of Organic Bananas, Organic Strawberries, Organic Baby Spinach, and Organic Hass Avocado** dominate purchase volume. Produce is also the largest department by volume. This makes raw purchase frequency a reasonable first baseline, but it is inherently non-personalized.

* **Purchase volume and reorder propensity are different signals.** Highly purchased products do not necessarily have the highest reorder rates. Some products with relatively modest purchase volume exhibit very high reorder rates. Therefore, reorder rate should not be used in isolation because low-volume products can produce artificially high rates.

* **The reorder-popularity baseline therefore requires a volume threshold.** The product purchase distribution is heavily right-skewed: the median product has **60 purchases**, while the 99th percentile has **9,931 purchases**. Using the 99th percentile as the minimum purchase threshold helps prevent low-volume products from dominating the reorder-based ranking.

* **Personalized frequency is expected to provide the strongest baseline signal.** Unlike the two global popularity approaches, Personalized Frequency uses each user's own purchase history. Given the strong repeat-purchase behavior observed in EDA, this approach should provide a substantially stronger benchmark for users with sufficient purchase history.

* **Evaluation is appropriately aligned with the recommendation task.** The **131,209 users with a labeled `train` basket** provide the offline evaluation population. Using the same users, Top-K setting, Precision@K, and Recall@K across all three baselines ensures that subsequent collaborative-filtering and hybrid models can be compared against consistent benchmarks.

Overall, the EDA supports a progression from **global popularity → reorder-aware popularity → personalized purchase frequency**. This provides an interpretable baseline ladder in which each model introduces an additional behavioral signal, allowing subsequent recommendation models to demonstrate whether their additional complexity produces meaningful improvement over simple historical signals.


In [9]:
results_df.to_csv(Path.cwd().parent / "data" / "processed" / "baseline_results.csv")